In [ ]:
import os
from typing import Any

import torch
from torch.ao.quantization import quantize_dynamic
import shapely
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import torchgeo.models
from torchgeo.datasets import Sentinel, stack_samples
from torchgeo.samplers import GridGeoSampler
from tqdm import tqdm

torch.multiprocessing.set_start_method('fork', force=True)

rasterio_best_practices = {
    'GDAL_DISABLE_READDIR_ON_OPEN': 'EMPTY_DIR',
    'AWS_NO_SIGN_REQUEST': 'YES',
    'GDAL_MAX_RAW_BLOCK_CACHE_SIZE': '200000000',
    'GDAL_SWATH_SIZE': '200000000',
    'VSI_CURL_CACHE_SIZE': '200000000',
}
os.environ.update(rasterio_best_practices)


class Sentinel2AnnualMosaic(Sentinel):
    # example filename 16SEA_B9_2024-01-01_2025-01-01.tif
    filename_glob = "*.tif"
    filename_regex = r"(?P<tile>\d{2}[A-Z]{3})_(?P<band>B\d{1,2}A?)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<stop>\d{4}-\d{2}-\d{2})\.tif"
    separate_files = True
    is_image = True
    date_format = '%Y-%m-%d'
    all_bands = (
        'B1',
        'B2',
        'B3',
        'B4',
        'B5',
        'B6',
        'B7',
        'B8',
        'B8A',
        'B9',
        'B11',
        'B12',
    )
    rgb_bands = ('B4', 'B3', 'B2')

    def plot(
        self,
        sample: dict[str, Any],
        show_titles: bool = True,
        suptitle: str | None = None,
    ) -> plt.Figure:
        rgb_indices = []
        for band in self.rgb_bands:
            if band in self.bands:
                rgb_indices.append(self.bands.index(band))
            else:
                raise ValueError(f"Band {band} not found in dataset bands: {self.bands}")

        image = sample['image'][rgb_indices].permute(1, 2, 0)
        image = (image / 3000).clip(0, 1)
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(image)
        ax.axis('off')

        if show_titles:
            ax.set_title('Image')

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig

root = "../16SEA"
dataset = Sentinel2AnnualMosaic(paths=root, res=10, cache=False)
sampler = GridGeoSampler(dataset, size=256, stride=128)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, num_workers=12, sampler=sampler, collate_fn=stack_samples)
print(len(dataset), len(sampler), len(dataloader))

quantize = False
compile = False
device = "mps"
dtype = torch.float16

12 43056 2691


In [3]:
model = torchgeo.models.RCF(in_channels=12, features=16, kernel_size=3, mode="gaussian")
model.eval()
model = model.to(device).to(dtype)

# Apply quantization if enabled
if quantize:
    model = quantize_dynamic(
        model, 
        {torch.nn.Linear, torch.nn.Conv2d},
        dtype=torch.qint8
    )
if compile:
    model = torch.compile(model, mode="reduce-overhead", fullgraph=True)

model


RCF()

In [ ]:
embeddings, geometries = [], []

with torch.inference_mode():
    model.eval()
    model = model
    for batch in tqdm(dataloader, total=len(dataloader)):
        image = batch["image"].to(device).to(dtype)
        emb = model(image).cpu().numpy()
        embeddings.append(emb)
        for bounds in batch["bounds"]:
            x, y, t = bounds
            centroid = ((x.start + x.stop) / 2, (y.start + y.stop) / 2)
            geom = shapely.geometry.Point(*centroid)
            geometries.append(geom)

embeddings = np.concatenate(embeddings, axis=0)

gdf = gpd.GeoDataFrame(
    data={"embedding": [emb.tolist() for emb in embeddings]},
    geometry=geometries,
    crs=dataset.crs,
)
gdf.to_crs(epsg=4326, inplace=True)
gdf.to_parquet("16SEA_embeddings.parquet")

In [ ]:
import geopandas as gpd

gdf = gpd.read_parquet("16SEA.parquet")
gdf

In [ ]:
gdf.explore()